# Task 2 — OLTP or OLAP?

**Ali Raza** · Roll No: **su92-bsdsm-f23-018** · Section: **BSDS-7A**
Data Warehouse & BI — Week 1

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#9aa4b2", "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937", "xtick.color": "#4b5563", "ytick.color": "#4b5563",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e7eb", "grid.linewidth": 0.7,
    "axes.axisbelow": True, "font.size": 10,
})
INK, BLUE, RED, GREY = "#1f2937", "#2563eb", "#dc2626", "#9ca3af"
os.makedirs("charts", exist_ok=True)
NOTE = "DWBI Week 1 · synthetic course data"


## The two-question test

**How many records does it touch?** and **right now, or over time?**
Few rows + right now → OLTP. Millions of rows + over time → OLAP.

In [2]:
rows = [
    (1, "Cashier scans a packet of chips and prints the bill", "OLTP", "one basket, right now — the sale must commit before the next customer"),
    (2, "CEO compares sales by category over the last 12 months", "OLAP", "millions of rows aggregated over a year — nobody is waiting at a till for it"),
    (3, "Customer updates her phone number in the store app", "OLTP", "exactly one row, updated now"),
    (4, "Marketing compares Eid week with a normal week, all branches", "OLAP", "two time windows aggregated across every branch"),
    (5, "Withdraw Rs 5,000 from an ATM", "OLTP", "one account balance, must be consistent this second"),
    (6, "Bank finds customers whose balance fell three months in a row", "OLAP", "a trend pattern scanned across customers and months"),
    (7, "Rider marks a food order as delivered", "OLTP", "one order's status flips, right now"),
    (8, "Head office ranks all branches by returns this quarter", "OLAP", "a quarter of returns aggregated and ranked"),
]
for n, s, label, why in rows:
    print(f"{n}. [{label}] {s}\n   → {why}\n")

1. [OLTP] Cashier scans a packet of chips and prints the bill
   → one basket, right now — the sale must commit before the next customer

2. [OLAP] CEO compares sales by category over the last 12 months
   → millions of rows aggregated over a year — nobody is waiting at a till for it

3. [OLTP] Customer updates her phone number in the store app
   → exactly one row, updated now

4. [OLAP] Marketing compares Eid week with a normal week, all branches
   → two time windows aggregated across every branch

5. [OLTP] Withdraw Rs 5,000 from an ATM
   → one account balance, must be consistent this second

6. [OLAP] Bank finds customers whose balance fell three months in a row
   → a trend pattern scanned across customers and months

7. [OLTP] Rider marks a food order as delivered
   → one order's status flips, right now

8. [OLAP] Head office ranks all branches by returns this quarter
   → a quarter of returns aggregated and ranked



## My own scenarios — from a Pakistani business I know (a Lahore bakery chain)

**OLTP:** *A cashier at the DHA branch voids one wrongly-scanned cake at 7pm.* One row, right now — the bill must correct before the customer pays. Obvious because nothing is aggregated and someone is waiting.

**OLAP:** *The owner compares cake-vs-biscuit sales across all 6 branches for the last 6 months to decide what to bake for Eid.* Millions of rows over time, read-only — obvious because it's a decision about the future, not a transaction happening now.

## A genuinely borderline scenario

**"The branch manager checks today's sales so far at 3pm."**

- *OLTP side:* it touches only today's rows at one branch, and the manager wants it *now* — it feels operational.
- *OLAP side:* it is still an **aggregation over a time window**, not a single-row read; and at 3pm the same query from 40 managers is 40 concurrent scans.

**Decision:** put it on the **OLAP side** (the warehouse, or at minimum a read replica). The two-question test decides it: the query aggregates over time, and OLTP's job is to keep the tills fast. A "quick peek" that scans is still a scan.

## The question to answer — why did the billing counters slow down?

An OLTP database is optimised for **fast writes**: row-level locks are taken and released in milliseconds, indexes serve single-row lookups, and the buffer pool holds hot rows. The CEO's 12-month report is a **long analytical scan** — it takes shared locks across huge ranges, floods the buffer pool with a year's history, and saturates I/O and CPU. Every cashier's `INSERT` then **queues behind the report's locks and I/O**. The tills didn't slow down by coincidence: the system was doing exactly what it was tuned for (writes), and the report stole the resources those writes need.